In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings


## Chroma DB:

from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

/home/jaywardhan/RAG_Udemy/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_9754/4098974645.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [5]:
sample_docs = [
    
"""
Machine Learning

Machine learning is a branch of artificial intelligence that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task. Machine learning algorithms analyze large amounts of data, identify patterns and relationships, and use the knowledge gained during training to make predictions on new or unseen data. The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning is used for tasks such as classification and regression, while unsupervised learning helps discover hidden patterns or groups within data. Reinforcement learning allows an agent to learn through interaction with an environment using rewards and penalties. Machine learning is widely used in applications such as recommendation systems, fraud detection, healthcare, image recognition, speech processing, autonomous vehicles, and financial forecasting.
""",

"""
Deep Learning and Neural Networks

Deep learning and neural networks are important areas of artificial intelligence that enable computers to learn complex patterns from large amounts of data. Neural networks are computational models inspired by the structure of the human brain and consist of interconnected neurons organized into input, hidden, and output layers. Deep learning uses neural networks with multiple hidden layers to automatically learn different levels of representations from data without requiring extensive manual feature extraction. During training, the network adjusts its weights and biases using algorithms such as backpropagation and gradient descent to improve its predictions. Deep learning and neural networks are widely used in image recognition, natural language processing, speech recognition, computer vision, autonomous vehicles, recommendation systems, and medical diagnosis. Common neural network architectures include Convolutional Neural Networks (CNNs), Recurrent Neural Networks (RNNs), and Transformer networks.
""",

"""
Natural Language Processing (NLP)

Natural Language Processing (NLP) is a branch of artificial intelligence that focuses on enabling computers to understand, interpret, process, and generate human language. NLP combines techniques from linguistics, computer science, and machine learning to work with large amounts of textual and spoken data. It is used for tasks such as text classification, sentiment analysis, language translation, speech recognition, named entity recognition, text summarization, and question answering. Traditional NLP methods include tokenization, stemming, lemmatization, and bag-of-words representations, while modern NLP systems use deep learning and transformer-based models to understand the context and relationships between words. NLP is widely used in applications such as virtual assistants, chatbots, search engines, recommendation systems, machine translation, and large language models.
""",


]

In [6]:
import tempfile

temp_dir = tempfile.mkdtemp()

for i , doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Documents created successfully in: {temp_dir}")

Documents created successfully in: /tmp/tmpt5ljkur3


In [7]:
print(f"Documents created successfully in: {temp_dir}")

Documents created successfully in: /tmp/tmpt5ljkur3


In [8]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    temp_dir,
    glob = "*.txt",
    loader_cls = TextLoader,
    loader_kwargs = {'encoding' : 'utf-8'}
)

txt_doc = loader.load()

print(len(txt_doc))
print(f"Doc1 preview: {txt_doc[0].page_content}")
print(f"Metadata: {txt_doc[0].metadata}")


3
Doc1 preview: 
Natural Language Processing (NLP)

Natural Language Processing (NLP) is a branch of artificial intelligence that focuses on enabling computers to understand, interpret, process, and generate human language. NLP combines techniques from linguistics, computer science, and machine learning to work with large amounts of textual and spoken data. It is used for tasks such as text classification, sentiment analysis, language translation, speech recognition, named entity recognition, text summarization, and question answering. Traditional NLP methods include tokenization, stemming, lemmatization, and bag-of-words representations, while modern NLP systems use deep learning and transformer-based models to understand the context and relationships between words. NLP is widely used in applications such as virtual assistants, chatbots, search engines, recommendation systems, machine translation, and large language models.

Metadata: {'source': '/tmp/tmpt5ljkur3/doc_2.txt'}


## Splitting text

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators = [" "]
)

chunks = text_splitter.split_documents(txt_doc)

In [10]:
print(len(chunks))

8


In [11]:
print(chunks[1])

page_content='recognition, named entity recognition, text summarization, and question answering. Traditional NLP methods include tokenization, stemming, lemmatization, and bag-of-words representations, while modern NLP systems use deep learning and transformer-based models to understand the context and relationships between words. NLP is widely used in applications such as virtual assistants, chatbots, search engines, recommendation systems, machine translation, and large language models.' metadata={'source': '/tmp/tmpt5ljkur3/doc_2.txt'}


In [12]:
txt_doc

[Document(metadata={'source': '/tmp/tmpt5ljkur3/doc_2.txt'}, page_content='\nNatural Language Processing (NLP)\n\nNatural Language Processing (NLP) is a branch of artificial intelligence that focuses on enabling computers to understand, interpret, process, and generate human language. NLP combines techniques from linguistics, computer science, and machine learning to work with large amounts of textual and spoken data. It is used for tasks such as text classification, sentiment analysis, language translation, speech recognition, named entity recognition, text summarization, and question answering. Traditional NLP methods include tokenization, stemming, lemmatization, and bag-of-words representations, while modern NLP systems use deep learning and transformer-based models to understand the context and relationships between words. NLP is widely used in applications such as virtual assistants, chatbots, search engines, recommendation systems, machine translation, and large language models.

## Initializing Chroma DB and stroing chunks in embedded or vector form

In [13]:
vector_directory = "./chroma_db"

## Creating chroma db:

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = OpenAIEmbeddings(),
    persist_directory = vector_directory,
    collection_name = "rag_collection" # vectorstore name
)

print(f"Vector Store created with {vectorstore._collection.count()} vectors")
print(f"Presistent Directory: {vector_directory}")

Vector Store created with 8 vectors
Presistent Directory: ./chroma_db


In [14]:
embedding = OpenAIEmbeddings()
print(embedding)

client=<openai.resources.embeddings.Embeddings object at 0x7445060e6fd0> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7445060e7b10> model='text-embedding-ada-002' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True


## Testing Similarity Search

In [15]:
query = "What are the types machine learning?"

similar_docs = vectorstore.similarity_search(query, k = 3)
similar_docs

print(f"Query: {query}\n")
for i , doc in enumerate(similar_docs):

    print(f"Relevant Doc: {i+1}")
    print(f"Content: {doc.page_content[:200]}...\n")
    print(f"Metadata: {doc.metadata}\n")

Query: What are the types machine learning?

Relevant Doc: 1
Content: Machine Learning

Machine learning is a branch of artificial intelligence that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task...

Metadata: {'source': '/tmp/tmpt5ljkur3/doc_0.txt'}

Relevant Doc: 2
Content: learning, and reinforcement learning. Supervised learning is used for tasks such as classification and regression, while unsupervised learning helps discover hidden patterns or groups within data. Rei...

Metadata: {'source': '/tmp/tmpt5ljkur3/doc_0.txt'}

Relevant Doc: 3
Content: different levels of representations from data without requiring extensive manual feature extraction. During training, the network adjusts its weights and biases using algorithms such as backpropagatio...

Metadata: {'source': '/tmp/tmpt5ljkur3/doc_1.txt'}



In [16]:
query = "What is Deep learning and what are it's uses?"

similar_docs = vectorstore.similarity_search(query, k = 3)
similar_docs
print(f"Query: {query}\n")
for i , doc in enumerate(similar_docs):

    print(f"Relevant Doc: {i+1}")
    print(f"Content: {doc.page_content[:200]}...\n")
    print(f"Metadata: {doc.metadata}\n")

Query: What is Deep learning and what are it's uses?

Relevant Doc: 1
Content: Deep Learning and Neural Networks

Deep learning and neural networks are important areas of artificial intelligence that enable computers to learn complex patterns from large amounts of data. Neural n...

Metadata: {'source': '/tmp/tmpt5ljkur3/doc_1.txt'}

Relevant Doc: 2
Content: different levels of representations from data without requiring extensive manual feature extraction. During training, the network adjusts its weights and biases using algorithms such as backpropagatio...

Metadata: {'source': '/tmp/tmpt5ljkur3/doc_1.txt'}

Relevant Doc: 3
Content: Machine Learning

Machine learning is a branch of artificial intelligence that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task...

Metadata: {'source': '/tmp/tmpt5ljkur3/doc_0.txt'}



### Advanced similarity search with scores

In [17]:
result_scores = vectorstore.similarity_search_with_score(query, k = 3)
result_scores

[(Document(metadata={'source': '/tmp/tmpt5ljkur3/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\nDeep learning and neural networks are important areas of artificial intelligence that enable computers to learn complex patterns from large amounts of data. Neural networks are computational models inspired by the structure of the human brain and consist of interconnected neurons organized into input, hidden, and output layers. Deep learning uses neural networks with multiple hidden layers to automatically learn different levels of representations from data'),
  0.27095019817352295),
 (Document(metadata={'source': '/tmp/tmpt5ljkur3/doc_1.txt'}, page_content='different levels of representations from data without requiring extensive manual feature extraction. During training, the network adjusts its weights and biases using algorithms such as backpropagation and gradient descent to improve its predictions. Deep learning and neural networks are widely used in image recognition,

In [18]:
## Initialize LLM, RAG chain, Prompt template, Query the RAG System

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name = "gpt-4o-mini",
    temperature = 0.2,
    max_tokens = 500
)

test_response = llm.invoke("What LLM uses NLP for response generation?")
print(test_response)

content="Large Language Models (LLMs) that utilize Natural Language Processing (NLP) for response generation include:\n\n1. **OpenAI's GPT Series**: Models like GPT-3 and GPT-4 are designed for generating human-like text based on prompts provided by users.\n\n2. **Google's BERT and T5**: While BERT is primarily used for understanding context in text, T5 (Text-to-Text Transfer Transformer) is designed for various NLP tasks, including text generation.\n\n3. **Facebook's BART**: BART (Bidirectional and Auto-Regressive Transformers) is effective for text generation and can be used for tasks like summarization and translation.\n\n4. **Microsoft's Turing-NLG**: This is a large-scale language model designed for generating natural language text.\n\n5. **Hugging Face's Transformers**: This library includes various models like DistilGPT-2, which are fine-tuned for specific NLP tasks, including response generation.\n\n6. **Anthropic's Claude**: A conversational AI model designed for generating re

In [19]:
## Other way of initializing:

from langchain.chat_models.base import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini")
type(llm)

langchain_openai.chat_models.base.ChatOpenAI

In [20]:
llm.invoke("Who was steve jobs?")

AIMessage(content='Steve Jobs (1955-2011) was an influential American entrepreneur, inventor, and business leader best known for co-founding Apple Inc. along with Steve Wozniak and Ronald Wayne in 1976. He played a pivotal role in revolutionizing several industries, including personal computing, music, smartphones, and digital publishing. \n\nJobs was instrumental in the development of groundbreaking products such as the Apple II, Macintosh, iPod, iPhone, and iPad. His vision combined technology with aesthetics, leading to innovative design and intuitive user interfaces that set Apple products apart from competitors.\n\nIn addition to his work at Apple, Jobs also co-founded Pixar Animation Studios, which produced famous animated films like "Toy Story" and "Finding Nemo." Under his leadership, Pixar transformed the animation industry.\n\nJobs is often recognized for his charismatic presentations and marketing strategies that built a loyal customer base for Apple. He served as Apple\'s C

## Creating Traditional RAG Chain

In [21]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [22]:
## Convert vector store to retriever

retriever = vectorstore.as_retriever(
    search_kwargs = {"k":3} ## Retrieve top 3 relevant chunks
)

retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x744507a75010>, search_kwargs={'k': 3})

In [23]:
system_prompt="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human","{input}")
])

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [24]:
## Creating a document chain that combines the context retrived from vector store one by one such that we can give it to the llm
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain 

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=No

In [25]:
### Creating final RAG chain:

from langchain_classic.chains import create_retrieval_chain
rag_chain = create_retrieval_chain(retriever,document_chain)

rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x744507a75010>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don'

In [26]:
response = rag_chain.invoke({"input": "What are the uses of machine learning?"})

In [27]:
response['answer']

'Machine learning is widely used in applications such as recommendation systems, fraud detection, healthcare, image recognition, speech processing, autonomous vehicles, and financial analysis. It enables computers to learn from data and make predictions or decisions across various domains. Its capabilities allow for improved accuracy and efficiency in tasks that require data analysis and pattern recognition.'

In [28]:
def response_of_queries(question):

    print(f"Query: {question}")
    print("-" * 50)

    resp = rag_chain.invoke({"input" : question})

    print(f"Answer: {resp['answer']}\n")
    print("Retrieved Documents: \n")
    for i , doc in enumerate(resp['context']):
        print(f"Source: {i+1}")
        print(f"Context: {doc.page_content[:200]}...")
        print(f"source of info: {doc.metadata.get('source')}\n")


text_query = [
    "What is the use of machine learning?",
    "What is Deep learning and what are it's usage?",
    "What is NLP?"
]

for query in text_query:
    response_of_queries(query)


Query: What is the use of machine learning?
--------------------------------------------------
Answer: Machine learning is used in various applications such as recommendation systems, fraud detection, healthcare, image recognition, speech processing, autonomous vehicles, and financial analysis. It enables computers to analyze large amounts of data, identify patterns, and make predictions or decisions without explicit programming for every task. This capability helps businesses and researchers improve efficiency and derive insights from data.

Retrieved Documents: 

Source: 1
Context: Machine Learning

Machine learning is a branch of artificial intelligence that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task...
source of info: /tmp/tmpt5ljkur3/doc_0.txt

Source: 2
Context: learning, and reinforcement learning. Supervised learning is used for tasks such as classification and regression, while unsupervised learning

In [29]:
response

{'input': 'What are the uses of machine learning?',
 'context': [Document(metadata={'source': '/tmp/tmpt5ljkur3/doc_0.txt'}, page_content='Machine Learning\n\nMachine learning is a branch of artificial intelligence that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task. Machine learning algorithms analyze large amounts of data, identify patterns and relationships, and use the knowledge gained during training to make predictions on new or unseen data. The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.'),
  Document(metadata={'source': '/tmp/tmpt5ljkur3/doc_0.txt'}, page_content='learning, and reinforcement learning. Supervised learning is used for tasks such as classification and regression, while unsupervised learning helps discover hidden patterns or groups within data. Reinforcement learning allows an agent to learn through interaction with an environmen

## Building RAG Using Langchain Expression Language

In [30]:
## Building a RAG Pipeline using Langchain Expression Laianguage(LCEL)

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel,RunnablePassthrough

In [31]:
# Creating a custom prompt:

custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on context, say you don't know.
Provied specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")

custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question.\nIf you don't know the answer based on context, say you don't know.\nProvied specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [32]:
# Formatting the documents returned by the retriever for the prompt

def format(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [36]:
# Creating RAG Chain using LCEL

rag_chain_lcel = (

    { "context" : retriever | format, 
      "question": RunnablePassthrough()
    }

    | custom_prompt
    | llm
    | StrOutputParser()
)

In [37]:
rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x744507a75010>, search_kwargs={'k': 3})
           | RunnableLambda(format),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question.\nIf you don't know the answer based on context, say you don't know.\nProvied specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07

In [ ]:
response = rag_chain_lcel.invoke("What is the use of deep learning?") # Whenever we use runnable pass through, we have to directly pass the question.There's no need to pass the question in a dictionary
response

'Deep learning is used in various applications, including image recognition, natural language processing, speech recognition, computer vision, autonomous vehicles, recommendation systems, and medical diagnosis. It enables computers to learn complex patterns from large amounts of data through neural networks that consist of interconnected neurons organized into layers, allowing for the automatic learning of different levels of representations from data.'

In [46]:
retriever.invoke("What is NLP?")

[Document(metadata={'source': '/tmp/tmpt5ljkur3/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\nNatural Language Processing (NLP) is a branch of artificial intelligence that focuses on enabling computers to understand, interpret, process, and generate human language. NLP combines techniques from linguistics, computer science, and machine learning to work with large amounts of textual and spoken data. It is used for tasks such as text classification, sentiment analysis, language translation, speech recognition, named entity recognition, text'),
 Document(metadata={'source': '/tmp/tmpt5ljkur3/doc_2.txt'}, page_content='recognition, named entity recognition, text summarization, and question answering. Traditional NLP methods include tokenization, stemming, lemmatization, and bag-of-words representations, while modern NLP systems use deep learning and transformer-based models to understand the context and relationships between words. NLP is widely used in applications such 

In [48]:
def lcel_format(question):

    print(f"Question: {question}\n")
    print("-" * 50)

    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    documents = retriever.invoke(question)

    for i,doc in enumerate(documents):
        print(f"Doc: {i+1}\n")
        print(f"Doc context: {doc.page_content}")
        print(f"Source: {doc.metadata.get("source")}\n")

lcel_format("What are the uses of Deep Learning?")
        


Question: What are the uses of Deep Learning?

--------------------------------------------------
Answer: Deep learning is widely used in various applications, including:

1. **Image recognition** - Deep learning models are employed to identify and classify objects within images.
2. **Natural language processing** - Techniques in deep learning are utilized to understand and generate human language.
3. **Speech recognition** - Deep learning algorithms help in converting spoken language into text.
4. **Computer vision** - This area leverages deep learning to enable machines to interpret and understand visual information from the world.
5. **Autonomous vehicles** - Deep learning plays a critical role in enabling self-driving cars to perceive their surroundings and make decisions.
6. **Recommendation systems** - These systems use deep learning to analyze user preferences and suggest relevant products or content.
7. **Medical diagnosis** - Deep learning technologies assist in analyzing medi

## Adding New Documents to existing vectorstore

In [54]:
new_document = """
    Reinforcement Learning:

    Reinforcement Learning (RL) is a type of machine learning in which an **agent learns to make decisions by interacting with an environment**. Instead of being trained using labeled data, the agent learns through **trial and error**. At each step, the agent observes the current state of the environment and chooses an action. After performing the action, it receives a **reward or penalty**, which indicates how good or bad the action was. The main objective of the agent is to learn a **policy** that helps it maximize the total reward over time. Reinforcement learning involves important concepts such as **states, actions, rewards, policies, and value functions**. For example, in a game, an RL agent can learn which moves are beneficial by receiving positive rewards for winning or progressing and negative rewards for losing or making poor decisions. Popular reinforcement learning algorithms include **Q-Learning, SARSA, Deep Q-Networks (DQN), and Policy Gradient methods**. Reinforcement Learning has applications in **robotics, autonomous vehicles, game playing, recommendation systems, resource management, and industrial automation**, where an intelligent system needs to learn effective decisions through continuous interaction with its environment.

"""

In [55]:
new_doc = Document(
    page_content = new_document,
    metadata = {"source": "Chat-GPT", "topic": "Reinforcement Learning"}
)

new_doc

Document(metadata={'source': 'Chat-GPT', 'topic': 'Reinforcement Learning'}, page_content='\n    Reinforcement Learning:\n\n    Reinforcement Learning (RL) is a type of machine learning in which an **agent learns to make decisions by interacting with an environment**. Instead of being trained using labeled data, the agent learns through **trial and error**. At each step, the agent observes the current state of the environment and chooses an action. After performing the action, it receives a **reward or penalty**, which indicates how good or bad the action was. The main objective of the agent is to learn a **policy** that helps it maximize the total reward over time. Reinforcement learning involves important concepts such as **states, actions, rewards, policies, and value functions**. For example, in a game, an RL agent can learn which moves are beneficial by receiving positive rewards for winning or progressing and negative rewards for losing or making poor decisions. Popular reinforce

In [56]:
## Splitting the text:
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'Chat-GPT', 'topic': 'Reinforcement Learning'}, page_content='Reinforcement Learning:\n\n    Reinforcement Learning (RL) is a type of machine learning in which an **agent learns to make decisions by interacting with an environment**. Instead of being trained using labeled data, the agent learns through **trial and error**. At each step, the agent observes the current state of the environment and chooses an action. After performing the action, it receives a **reward or penalty**, which indicates how good or bad the action was. The main objective of the'),
 Document(metadata={'source': 'Chat-GPT', 'topic': 'Reinforcement Learning'}, page_content='or bad the action was. The main objective of the agent is to learn a **policy** that helps it maximize the total reward over time. Reinforcement learning involves important concepts such as **states, actions, rewards, policies, and value functions**. For example, in a game, an RL agent can learn which moves are bene

In [57]:
## Add new documents to vector store:

vectorstore.add_documents(new_chunks)

['c283fac8-7ed9-4625-8094-2ef71ece842e',
 'a9e74b06-cfd8-43e1-a703-83db2822546c',
 '7df02057-894d-4b09-ab1c-f6a219c644b1']

In [58]:
print(f"Total vectors in vectorstore after adding new vectors are: {vectorstore._collection.count()}")

Total vectors in vectorstore after adding new vectors are: 11


In [61]:
## query with the updated vector:

new_question = "What is Reinforcement Learning in short and what are it's use cases?"
lcel_format(new_question)

Question: What is Reinforcement Learning in short and what are it's use cases?

--------------------------------------------------
Answer: Reinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by interacting with an environment through trial and error. The agent observes the current state, chooses actions, and receives rewards or penalties based on those actions, aiming to maximize overall reward.

Use cases for reinforcement learning include:
- Robotics
- Autonomous vehicles
- Game playing
- Recommendation systems
- Resource management
- Industrial automation

These applications benefit from the agent's ability to learn effective decision-making through continuous interaction with the environment.
Doc: 1

Doc context: Reinforcement Learning:

    Reinforcement Learning (RL) is a type of machine learning in which an **agent learns to make decisions by interacting with an environment**. Instead of being trained using labeled data, the agent le

## Advanced RAG Techniques: Conversational Memory

1. create_history_aware_retriever: Makes the retriever understand the conversation context
2. MessagesPlaceholder: Placeholder for chat history in prompts
3. HumanMessage/AIMessage: Structured message types for conversation history

In [62]:
## Conversational Memory: Adding Previous Conversational Context 
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [63]:
## creating a prompt that includes the chat history

contextualize_q_system_prompt = """Given a chat history and the latest user question
which might reference context in the chat history, formulate a standalone question
which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is"""

In [64]:
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

In [65]:
## Creating a history aware retriever

history_aware_retriever = create_history_aware_retriever(
    llm,retriever,contextualize_q_prompt
)

history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x744507a75010>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessag

In [66]:
## Create a new document chain with history

qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])

question_answer_chain = create_stuff_documents_chain(llm,qa_prompt)

In [67]:
## Creating Conversational RAG Chain

conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)

In [68]:
chat_history = []

result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})

print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a branch of artificial intelligence that allows computers to learn from data and make predictions or decisions without being explicitly programmed for every task. It involves analyzing large amounts of data to identify patterns and relationships, enabling the system to make predictions on new or unseen data. The main types of machine learning include supervised learning, unsupervised learning, and reinforcement learning.


In [70]:
chat_history.extend([
    HumanMessage(content = "What is machine learning?"),
    AIMessage(content = result1['answer'])
])

result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are it's types?"
})

print(f"Q: What are it's types?")
print(f"A: {result2['answer']}")


Q: What are it's types?
A: The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning is used for tasks such as classification and regression, while unsupervised learning helps discover hidden patterns or groups within data. Reinforcement learning allows an agent to learn through interaction with an environment using rewards and penalties.


## Using Groq's LLM

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [7]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

groq_llm = ChatGroq(model = "meta-llama/llama-prompt-guard-2-22m")

groq_llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama Prompt Guard 2 22M', 'release_date': '2024-10-01', 'last_updated': '2024-10-01', 'open_weights': True, 'max_input_tokens': 512, 'max_output_tokens': 512, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7738b6c5c440>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7738b6c5cec0>, model_name='meta-llama/llama-prompt-guard-2-22m', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [10]:
groq_llm = init_chat_model("groq:qwen/qwen3.6-27b")
groq_llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x7738b6cf6ad0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7738b6cf74d0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [11]:
groq_llm.invoke("Who was steve jobs?")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Identify the core entity**: The user is asking about "Steve Jobs".\n2.  **Determine key facts needed**: \n   - Full name: Steven Paul Jobs\n   - Lifespan: February 24, 1955 – October 5, 2011\n   - Primary role: Co-founder, chairman, and former CEO of Apple Inc.\n   - Key contributions: Played a pivotal role in personal computer revolution, digital music, online music stores, tablet computing, digital publishing, and mobile telephony.\n   - Major products: Apple I, Apple II, Macintosh, iPod, iTunes Store, iPhone, iPad, Mac OS X, iOS.\n   - Other ventures: NeXT Computer (founded after leaving Apple), Pixar Animation Studios (founded, later sold to Disney).\n   - Leadership style/philosophy: Known for design-focused approach, innovation, marketing, and "Think Different" campaign.\n   - Legacy: Widely regarded as one of the most influential figures in technology and business.\n3.  **Structure the response**: \n   - Introduc